In [1]:
from typing import List,TypedDict,Literal
from pydantic import BaseModel
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv

from langchain_community.tools.tavily_search import TavilySearchResults
load_dotenv()

True

In [2]:
!pip install pypdf

Defaulting to user installation because normal site-packages is not writeable


In [3]:
 docs=(
        PyPDFLoader("./Documents/buildLLM.pdf").load()
      + PyPDFLoader("./Documents/handsonLLM.pdf").load()
    )

In [4]:
len(docs)

945

In [6]:
chunks=RecursiveCharacterTextSplitter(chunk_size=900,chunk_overlap=150).split_documents(docs)
# clean text for avoiding unicodeEncoder Errr
for d in chunks:
    d.page_content=d.page_content.encode("utf-8","ignore").decode("utf-8","ignore")

In [7]:
len(chunks)

2072

In [8]:
!pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable


In [9]:
pip install faiss-cpu

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Swap OpenAI for a free, highly capable open-source model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Create your vector store and retriever (this runs locally, no API key needed!)
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 4})

print("Success! Vector store created locally.")

C:\Users\DELL\AppData\Local\Temp\ipykernel_15896\678481376.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Success! Vector store created locally.


In [ ]:
# embeddings=OpenAIEmbeddings(model='text-embedding-3-large')
# vector_store=FAISS.from_documents(chunks,embeddings)
# retriever= vector_store.as_retriever(search_type='similarity',search_kwargs={'k':4})

In [11]:
# llm=ChatOpenAI(model="gpt-4o-mini",temperature=0)

In [12]:
pip install -U langchain-ollama

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [11]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model = "llama3",temperature=0)

In [12]:
UPPER_TH=0.7
LOWER_TH=0.3


In [13]:
class State(TypedDict):
    question: str
    docs: List[Document]

    good_docs:List[Document]
    verdict: str
    reason: str

    strips: List[str]
    kept_strips:List[str]
    refined_context: str

    web_docs:List[Document]

    web_query:str
    
    answer:str

In [14]:
def retrieve_node(state:State)->State:
    q=state['question']
    return {"docs":retriever.invoke(q)}

In [15]:
# score based doc evaluator

class DocEvalScore(BaseModel):
    score:float
    reason:str
doc_eval_prompt=ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "you are strict retrieval evaluator for RAG.\n"
            "you will be given ONE retrieved chunk and a question.\n"
            "return a relevance score in [0.0,1.0].\n"
            "-1.0: chunk alone is sufficient to answer fully/mostly\n"
            "-0.0: chunk is irrelevant\n"
            "Be conservative with high scores.\n"
            "Also return a short reason.\n"
            "Output JSON only.",
        ),
        ("human","Question:{question}\n\nChunk:{chunk}"),
    ]
)
doc_eval_chain = doc_eval_prompt | llm.with_structured_output(DocEvalScore)

def  eval_each_doc_nodes(state:State)->State:
    q = state["question"]
    scores: List[float] = []
    good: List[Document] = []
    
    for d in state["docs"]:
        out = doc_eval_chain.invoke({"question":q,"chunk":d.page_content})
        scores.append(out.score)
        if out.score > LOWER_TH:
            good.append(d)
    if any(s > UPPER_TH for s in scores):
        return{
            "good_docs":good,
            "verdict":"CORRECT",
            "reason":f"At least one retrieved chunk scored > {UPPER_TH}.",
        }
    if len(scores) > 0 and all(s < LOWER_TH for s in scores):
        why = "N0 chunk was sufficient."
        return{
            "good_docs":[],
            "verdict":"INCORRECT",
            "reason":f"All retrieved chunk scored < {LOWER_TH}.{why}",
        }
    why = "Mixed relevance signals."
    return {
        "good_docs":good,
        "verdict": "AMBIGUOUS",
        "reason":f"No chunk scored > {UPPER_TH}, but not all were < {LOWER_TH}. {why}",
    }

In [16]:
# sentence level Decomposer

def decompose_to_sentences(text: str)->List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+",text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]

# filter LLM judge
class KeepOrDrop(BaseModel):
    keep:bool
filter_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a strict relevance filter.\n"
            "Return keep = true only if the sentence directly helps answer the question.\n"
            "Use ONLY the sentence. Output JSON only.",
        ),
        ("human","Question: {question}\n\nSentence:\n{sentence}"),
    ]
)

filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop)

# Refining( decompose -> Filter->Recompose)

def refine(state:State)->State:
    q = state["question"]
    if state.get("verdict")=="CORRECT":
        context = "\n\n".join(d.page_content for d in state["good_docs"]).strip()
    else:
        context = "\n\n".join(d.page_content for d in state["web_docs"]).strip()
    
    strips = decompose_to_sentences(context)
    
    #filter: keeping only relevant stips
    kept: List[str] = []
    for s in strips:
        if filter_chain.invoke({"question":q,"sentence":s}).keep:
            kept.append(s)
    # Recompose: keep together (internal knowledge)
    refined_context = "\n".join(kept).strip()
    return{
        "strips":strips,
        "kept_strips":kept,
        "refined_context":refined_context,
    }

In [17]:
class WebQuery(BaseModel):
    query:str
rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Rewrite the user question into a web search query composed of keywords. \n"
            "Rules:\n"
            "- Keep it short (6-14 words).\n"
            "-if the question implies recency (e.g.,recent/latest/last week/last month), add a constraint like (last)"
            "-Do not answer the question.\n"
            "- Return JSON with a single key: query",
        ),
        ("human","Question: {question}"),
    ]
)

rewrite_chain=rewrite_prompt | llm.with_structured_output(WebQuery)

def rewrite_query_node(state: State)->State:
    out = rewrite_chain.invoke({"question":state["question"]})
    return {"web_query":out.query}

# web search node: use web query 

# tavily = TavilySearchResults(max_results=5)

tavily = TavilySearchResults(
    max_results=5, 
    tavily_api_key="tvly-your-actual-key-here"
)
def web_search_node(state: State)-> State:
    q = state.get("web_query") or state["question"]
    result = tavily.invoke({"query":q})

    web_docs = []
    
    for r in result or []:
        title = r.get("title","")
        url = r.get("url","")
        content = r.get("content","") or r.get("snippet","")
        text = f"TITLE:{title}\nURL:{url}\nCONTENT:\n{content}"
        web_docs.append(Document(page_content=text,metadata={"url":url,"title":title}))
    return {"web_docs":web_docs}


C:\Users\DELL\AppData\Local\Temp\ipykernel_15896\3956428542.py:28: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults(


In [18]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful ML tutor. Answer ONLY using the provided context.\n"
            "if the context is empty or insufficient, say:'I don't know.'",
        ),
        ("human","Question:{question}\n\nRefined context:\n{refined_context}"),
    ]
)

def generate(state:State)->State:
    out = (answer_prompt | llm).invoke(
        {"question": state["question"], "refined_context":state["refined_context"]}
    )
    return {"answer":out.content}

In [19]:
def fail_node(state:State)->State:
    return {"answer":f"FAIL: {state["reason"]}"}

def ambiguous_node(state:State)->State:
    return {"answer":f"Ambiguous:{state["reason"]}"}

def route_after_eval(state:State)->str:
    if state["verdict"] =="CORRECT":
        return "refine"
    elif state["verdict"] == "INCORRECT":
        return "web_search"
    else:
        return "ambiguous"

In [20]:
g = StateGraph(State)

# Add all nodes correctly
g.add_node("retrieve", retrieve_node)
g.add_node("eval_each_doc", eval_each_doc_nodes)
g.add_node("rewrite_query", rewrite_query_node)
g.add_node("web_search", web_search_node)
g.add_node("refine", refine)
g.add_node("generate", generate) # Fixed syntax: comma instead of colon
g.add_node("ambiguous", ambiguous_node) # Added missing node

# Define edges
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "eval_each_doc")

g.add_conditional_edges(
    "eval_each_doc",
    route_after_eval,
    {
        "refine": "refine",
        "web_search": "rewrite_query", # Fixed: Matches route_after_eval return string
        "ambiguous": "ambiguous",
    },
)

g.add_edge("rewrite_query", "web_search")
g.add_edge("web_search", "refine")
g.add_edge("refine", "generate")
g.add_edge("generate", END)
g.add_edge("ambiguous", END)

app = g.compile()


# g= StateGraph(State)
# g.add_node("retrieve",retrieve_node)
# g.add_node("eval_each_doc", eval_each_doc_nodes)

# g.add_node("rewrite_query", rewrite_query_node)
# g.add_node("web_search", web_search_node)

# g.add_node("refine",refine)
# g.add_node("generate",generate)

# g.add_edge(START,"retrieve")
# g.add_edge("retrieve", "eval_each_doc")

# g.add_conditional_edges(
#     "eval_each_doc",
#     route_after_eval,
#     {
#         "refine":"refine",
#         "rewrite_query":"rewrite_query",
#         "ambiguous":"ambiguous",
#     },
# )
# # Incorrect path :rewrite web_search ->refine->generate
# g.add_edge("rewrite_query","web_search")
# g.add_edge("web_search", "refine")
# g.add_edge("refine","generate")

# g.add_edge("generate",END)
# g.add_edge("ambiguous",END)

# app=g.compile()
# app

# # g.add_node(START,"retrieve")
# # g.add_node("retrieve","generate")

# # g.add_node("generate", END)
# # app=g.compile()

# # app

In [21]:
res = app.invoke(
    {
        "question":"Recent AI news",
        "docs":[],
        "good_docs":[],
        "verdict":"",
        "reason":"",
        "strips":[],
        "kept_strips":[],
        "refined_context":"",
        "web_docs":[],
        "answer":"",
    }
)

print("VERDICT:",res["verdict"])
print("REASON:",res["reason"])
print("\nOUTPUT:\n",res["answer"])

ConnectError: [WinError 10061] No connection could be made because the target machine actively refused it

In [34]:
res["web_query"]

NameError: name 'res' is not defined

In [30]:

def retriever(state):
    q=state["question"]
    return {"docs":retriever.invoke(q)}

In [31]:
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","Answer only from context. if not in context, say dont know."),
        ("human","Question:{question}\n\nContext:\n{context}"),
    ]
)
def generate(state):
    context="\n\n".join(d.page_content for d in state[docs])
    out=(prompt | llm).invoke({"question":state["question"],"context":context})
    return {"answer":out.content}

In [ ]:
res=app.invoke(
    {
        "question":"Explain the bias-variance tradeoff",
        "docs":[],
        "strips":[],
        "kept_strips":[],
        "refined_context":"",
        "answer":""
    })
print(res["answer"])

AttributeError: 'function' object has no attribute 'invoke'

In [35]:
res["web_query"]

NameError: name 'res' is not defined

In [ ]:
print(res['docs'][0].page_content)